In [2]:
import json
import ollama

In [4]:
model = "llama3.2:latest"

In [5]:
def user(content):
    return {
        'role':'user',
        'content':content
            }
def system(content):
    return {
        'role':'system',
        'content':content
            }
def assistent(content):
    return {
        'role':'assistent',
        'content':content
            }

#### Defining tool functions

In [6]:
def get_current_temperature(location):
    """
    Simulates getting the current temperature for a given location.
    In a real app, this would call a weather API.
    """
    # Simulated data for demonstration
    temperatures = {
        "london": 15,
        "paris": 22,
        "new york": 28
    }
    return temperatures.get(location.lower(), 20)

def get_rain_chance(location):
    """
    Simulates getting the chance of rain for a given location.
    """
    rain_data = {
        "london": 70,
        "paris": 10,
        "new york": 40
    }
    return rain_data.get(location.lower(), 25)


#### defining tools

In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_temperature",
            "description": "Get the current temperature for a specific location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g., San Francisco, CA"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_rain_chance",
            "description": "Get the chance of rain (%) for a specific location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g., San Francisco, CA"
                    }
                },
                "required": ["location"]
            }
        }
    }
]


#### AI using function

In [8]:
def ai(msg,tools=None):
    return ollama.chat(
        model=model,
        messages=msg,
        tools=tools
        )

#### system prompt

In [9]:
# prompt = """
# You are an interviewer at a big tech company and user is here to take the interview for Full Stack Developer Intern
# """

In [10]:
prompt = """
You are a help ful assistent
"""

In [46]:
messages=[]

In [47]:
messages.append(system(prompt))

In [48]:
messages.append(user('''
                     can u write me python code to get current date
                     '''))

In [49]:
messages

[{'role': 'system', 'content': '\nYou are a help ful assistent\n'},
 {'role': 'user',
  'content': '\n                     can u write me python code to get current date\n                     '}]

In [50]:
response = ai(messages)

In [52]:
print(response.message.content)

Here is an example of Python code that prints the current date:

```python
from datetime import datetime

def get_current_date():
    # Get the current date and time
    now = datetime.now()
    
    # Format the date as YYYY-MM-DD
    formatted_date = now.strftime("%Y-%m-%d")
    
    return formatted_date

# Call the function to print the current date
current_date = get_current_date()
print(current_date)
```

This code uses the `datetime` module in Python, which provides classes for manipulating dates and times. The `now()` method returns a `datetime` object representing the current local date and time, and the `strftime()` method formats this date as a string according to a specified format.

You can replace `"%Y-%m-%d"` with any other format code from the [strftime() documentation](https://docs.python.org/3/library/datetime.html#time-format-code) to get the date in a different format (e.g. `%Y` for YYYY, `%m` for MM, etc.).

Alternatively, you can use the `date()` function from the

In [33]:
messages.append(assistent(response.message.content))

In [34]:
messages

[{'role': 'system', 'content': '\nYou are a help ful assistent\n'},
 {'role': 'user',
  'content': "\n                     What's the weather like in London right now? I need the temperature and the chance of rain.\n                     "},
 {'role': 'assistent', 'content': ''}]

### for tool calling

In [45]:
if response.message.tool_calls:
    print(f"🧠 Model is requesting to call {len(response.message.tool_calls)} tool(s):")
    
    # Dictionary to store results
    tool_results = {}
    
    # Process all tool calls (parallel execution)
    for tool in response.message.tool_calls:
        function_name = tool.function.name
        arguments = tool.function.arguments  # Already a dict in recent Ollama
        
        print(f"   🔧 Calling: {function_name} with {arguments}")
        
        # Execute the tool based on its name
        if function_name == "get_current_temperature":
            result = get_current_temperature(**arguments)
        elif function_name == "get_rain_chance":
            result = get_rain_chance(**arguments)
        else:
            result = f"Unknown tool: {function_name}"
        
        # Store result
        tool_results[function_name] = str(result)
    
    print("\n📊 Tool results:", tool_results)
    
    # ============================================================================
    # 6. Second API call: Send tool results back to Ollama for final response
    # ============================================================================
    # Prepare messages for final response
    final_messages = messages.copy()
    final_messages.append(response.message)  # Add the assistant's tool call response
    final_messages.append({
        "role": "tool",
        "content": json.dumps(tool_results)
    })
    
    print("\n📤 Sending tool results back to Ollama...")
    final_response = ollama.chat(
        model='llama3.2:latest',
        messages=final_messages
    )
    
    print("\n🤖 Final Answer:", final_response.message.content)
else:
    # If no tool is needed, just print the response
    print("🤖 Response:", response.message.content)


🧠 Model is requesting to call 2 tool(s):
   🔧 Calling: get_current_temperature with {'location': 'London, UK'}
   🔧 Calling: get_rain_chance with {'location': 'London, UK'}

📊 Tool results: {'get_current_temperature': '20', 'get_rain_chance': '25'}

📤 Sending tool results back to Ollama...

🤖 Final Answer: The current temperature in London is 20°C (68°F). There is a 25% chance of rain today. Please note that this information is based on my training data, and the accuracy may vary depending on the specific location within London and any recent changes to weather conditions. For the most up-to-date and accurate forecast, I recommend checking a reliable weather website or app, such as BBC Weather or AccuWeather.


In [56]:
final_messages


[{'role': 'system', 'content': '\nYou are a help ful assistent\n'},
 {'role': 'user',
  'content': "\n                     What's the weather like in London right now? I need the temperature and the chance of rain.\n                     "},
 Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_current_temperature', arguments={'location': 'London, UK'})), ToolCall(function=Function(name='get_rain_chance', arguments={'location': 'London, UK'}))]),
 {'role': 'tool',
  'content': '{"get_current_temperature": "20", "get_rain_chance": "25"}'}]

### AI WITH PARAMS

In [66]:
import ollama

# Simple chat with options
response = ollama.chat(
    model='llama3.2:latest',
    messages=[{"role": "user", "content": "write an essay of 300 words on Computer"
    ""}],
    options={
        "temperature": 2,        # More focused
        "top_p": 1,              # Nucleus sampling
        "num_predict": 300,        # Max 300 tokens
        "num_ctx": 4096,           # Larger context
        "repeat_penalty": 1.1,     # Less repetition
        "seed": 123,               # Reproducible output
        "mirostat": 1              # Advanced sampling
    },
    keep_alive="5m"  # Keep in memory
)

print(response)

model='llama3.2:latest' created_at='2026-08-27T14:52:49.8209577Z' done=True done_reason='stop' total_duration=1784196600 load_duration=202717800 prompt_eval_count=34 prompt_eval_duration=313447100 eval_count=90 eval_duration=1139904300 message=Message(role='assistant', content='The computer has been an essential part of modern life for several decades, and its impact continues to grow exponentially. These devices are versatile and have influenced multiple domains such as education, medical diagnosis and personal communications among others.\n\nDevelopped in the early part, it allowed us humans to work fast during wars like WWII etc, computers can also automate tasks. It provides information whenever anybody require and store a ton of useful thing about computer using an external storage media', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [67]:
response.message.content

'The computer has been an essential part of modern life for several decades, and its impact continues to grow exponentially. These devices are versatile and have influenced multiple domains such as education, medical diagnosis and personal communications among others.\n\nDevelopped in the early part, it allowed us humans to work fast during wars like WWII etc, computers can also automate tasks. It provides information whenever anybody require and store a ton of useful thing about computer using an external storage media'

### New Idea Here